# 01 · Download VizWiz data (Step 1.2)

**Runtime: CPU** (Runtime → Change runtime type → CPU, High RAM if offered). No GPU needed, so no GPU credits used.

What it does:
1. Downloads ~19 GB from the official VizWiz server into `MyDrive/vizwiz/datasets/raw/`: 3 image zips, annotations, and the official eval code.
2. Checks every file's size, records a SHA-256 checksum, and counts the images inside each zip.
3. Times unzipping the **val** images from Drive to local disk, which decides how we store images.

**If Colab disconnects:** reconnect and run all cells again. Finished files are skipped and partial ones resume.

Run cells top to bottom. Expect roughly **30–90 minutes**, depending on the server's speed.

## 1 · Settings, Drive, code

In [ ]:
from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"  # persistent: zips + checksums
LOCAL = "/content/data"  # fast local disk: unzipped images (wiped on disconnect)
LOGS = f"{DRIVE_ROOT}/logs"

drive.mount("/content/drive")
!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} log --oneline -1
!pip install -q -e {REPO_DIR}

**Paste back:** the commit line.

## 2 · Download (the long part)
Prints progress every 30 s per file. The small annotation files go first, then val, test, train.

In [ ]:
%%time
!python -m vizwiz.data.download fetch --root {DATA}

**Paste back:** the last ~15 lines (the `Zip contents:` block and `ALL OK`) and the `Wall time` line. If it fails, paste the full error.

## 3 · Time unzipping val images to local disk
Unzips the 7,750 val images (3.5 GB) from Drive to `/content/data`. The time tells us whether unzipping each session is fast enough.

In [ ]:
%%time
!python -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits val
!df -h /content | tail -1

**Paste back:** the JSON timing report and the `Wall time` line.

## 4 · Save reports to the logs folder

In [ ]:
!cp {DATA}/checksums.json {LOGS}/checksums.json
!cp {DATA}/extract_timings.json {LOGS}/extract_timings.json
!ls -la {DATA}/raw

**Upload back:** `MyDrive/vizwiz/logs/checksums.json` and `MyDrive/vizwiz/logs/extract_timings.json`, plus the `ls` output.

## 5 · (Optional) Check your W&B key
Only if you have already added `WANDB_API_KEY` in Colab Secrets (🔑 icon in the left sidebar, with **Notebook access** on). The key is read from Secrets and never printed.

In [ ]:
!pip install -q wandb
import wandb
from google.colab import userdata

ok = wandb.login(key=userdata.get("WANDB_API_KEY"), verify=True, relogin=True)
print("W&B login OK" if ok else "W&B login FAILED")

## 6 · Flush Drive before disconnecting
Makes sure every byte written to Drive is uploaded before the runtime goes away. Run this last, then disconnect (Runtime → Disconnect and delete runtime).

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")